# AlphaMissense annotation for CFTR variants

Requires `cftr2_results.csv` from `cftr2_scraper.ipynb`.

In [1]:
import os
import re
import gzip
import requests
import pandas as pd

## Step 1. Download AlphaMissense

In [2]:
AM_URL = "https://zenodo.org/records/8208688/files/AlphaMissense_hg38.tsv.gz"
AM_FILE = "AlphaMissense_hg38.tsv.gz"

if os.path.exists(AM_FILE):
    print(f"Already downloaded: {AM_FILE}")
else:
    print("Downloading AlphaMissense hg38 (~1.4 GB)...")
    with requests.get(AM_URL, stream=True, timeout=600) as resp:
        resp.raise_for_status()
        total = 0
        with open(AM_FILE, "wb") as f:
            for chunk in resp.iter_content(chunk_size=10 * 1024 * 1024):
                f.write(chunk)
                total += len(chunk)
                print(f"\r  {total / 1e9:.2f} GB", end="")
    print(f"\nDone. {os.path.getsize(AM_FILE) / 1e9:.2f} GB saved.")

Already downloaded: AlphaMissense_hg38.tsv.gz


## Step 2. Filter to CFTR

In [3]:
CFTR_UNIPROT = "P13569"

rows = []
header = None

print("Filtering to CFTR (P13569)...")
with gzip.open(AM_FILE, "rt", encoding="utf-8") as f:
    for line in f:
        if line.startswith("##"):
            continue
        if line.startswith("#CHROM"):
            header = line.strip().lstrip("#").split("\t")
            continue
        parts = line.strip().split("\t")
        if len(parts) > 5 and parts[5] == CFTR_UNIPROT:
            rows.append(parts)

cftr_am = pd.DataFrame(rows, columns=header)
cftr_am.to_csv("cftr_alphamissense.tsv", sep="\t", index=False)
print(f"Done. Saved {len(cftr_am)} CFTR AlphaMissense scores.")
print(cftr_am[["protein_variant", "am_pathogenicity", "am_class"]].head())

Filtering to CFTR (P13569)...
Done. Saved 9721 CFTR AlphaMissense scores.
  protein_variant am_pathogenicity       am_class
0             Q2E           0.1434  likely_benign
1             Q2K            0.183  likely_benign
2             Q2R           0.1896  likely_benign
3             Q2L           0.2826  likely_benign
4             Q2P           0.2483  likely_benign


## Step 2. Convert three-letter to single-letter amino acid codes

AlphaMissense uses single-letter format (`S13F`). Our VCF variants are three-letter (`Ser13Phe`). Convert to match.

In [4]:
AA3TO1 = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D',
    'Cys': 'C', 'Glu': 'E', 'Gln': 'Q', 'Gly': 'G',
    'His': 'H', 'Ile': 'I', 'Leu': 'L', 'Lys': 'K',
    'Met': 'M', 'Phe': 'F', 'Pro': 'P', 'Ser': 'S',
    'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'
}

def three_to_one(variant: str) -> str | None:
    """Convert Ser13Phe -> S13F. Returns None if conversion fails."""
    m = re.match(r'([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})', variant)
    if not m:
        return None
    ref, pos, alt = m.groups()
    if ref not in AA3TO1 or alt not in AA3TO1:
        return None
    return f"{AA3TO1[ref]}{pos}{AA3TO1[alt]}"

# Load results and convert
result = pd.read_csv("cftr2_results.csv")
result['am_variant'] = result['variant'].apply(three_to_one)

failed = result['am_variant'].isna().sum()
print(f"Converted: {len(result) - failed} / {len(result)}")
print(f"Failed   : {failed}")
print()
print(result[['variant', 'am_variant']].head(5))

Converted: 2909 / 3220
Failed   : 311

    variant am_variant
0  Ser13Phe       S13F
1  Arg31Cys       R31C
2  Arg31Leu       R31L
3  Ser42Phe       S42F
4  Asp44Gly       D44G


In [5]:
cftr_am = pd.read_csv("cftr_alphamissense.tsv", sep="\t")

result = result.merge(
    cftr_am[["protein_variant", "am_pathogenicity", "am_class"]],
    left_on="am_variant",
    right_on="protein_variant",
    how="left"
).drop(columns=["protein_variant"])

result.to_csv("cftr2_results_annotated.csv", index=False)

print(f"AlphaMissense scored : {result['am_pathogenicity'].notna().sum()}")
print(f"Not scored           : {result['am_pathogenicity'].isna().sum()}")
print()
print(result[["variant", "am_variant", "determination_2026", "am_pathogenicity", "am_class"]].head(10))


AlphaMissense scored : 3161
Not scored           : 555

    variant am_variant            determination_2026  am_pathogenicity  \
0  Ser13Phe       S13F                    CF-causing            0.7624   
1  Arg31Cys       R31C                Non CF-causing            0.1079   
2  Arg31Leu       R31L  Varying clinical consequence            0.1732   
3  Ser42Phe       S42F   No interpretation available            0.1165   
4  Asp44Gly       D44G                           NaN            0.5282   
5  Ser50Tyr       S50Y                           NaN            0.8651   
6  Trp57Gly       W57G                    CF-causing            0.9560   
7  Pro67Leu       P67L                    CF-causing            0.6516   
8  Arg74Trp       R74W  Varying clinical consequence            0.1405   
9  Arg75Gln       R75Q                Non CF-causing            0.1994   

            am_class  
0  likely_pathogenic  
1      likely_benign  
2      likely_benign  
3      likely_benign  
4          amb

In [6]:
from sklearn.metrics import roc_auc_score, classification_report
import numpy as np

# Only use variants with both CFTR2 determination and AM score
validated = result[
    result["determination_2026"].notna() &
    result["am_pathogenicity"].notna()
].copy()

# Binary label: CF-causing = 1, Non CF-causing = 0, drop Varying/No interpretation
binary = validated[
    validated["determination_2026"].isin(["CF-causing", "Non CF-causing"])
].copy()

binary["label"] = (binary["determination_2026"] == "CF-causing").astype(int)

auc = roc_auc_score(binary["label"], binary["am_pathogenicity"])
print(f"Variants used : {len(binary)}")
print(f"CF-causing    : {binary['label'].sum()}")
print(f"Non CF-causing: {(binary['label'] == 0).sum()}")
print(f"AUC           : {auc:.3f}")
print()
print(classification_report(
    binary["label"],
    (binary["am_pathogenicity"] >= 0.5).astype(int),
    target_names=["Non CF-causing", "CF-causing"]
))


Variants used : 292
CF-causing    : 253
Non CF-causing: 39
AUC           : 0.946

                precision    recall  f1-score   support

Non CF-causing       0.77      0.77      0.77        39
    CF-causing       0.96      0.96      0.96       253

      accuracy                           0.94       292
     macro avg       0.87      0.87      0.87       292
  weighted avg       0.94      0.94      0.94       292



In [7]:
unmatched = result[
    result["determination_2026"].isna() &
    result["am_pathogenicity"].notna()
].copy()

print(f"Unmatched variants with AM score: {len(unmatched)}")
print()
print(unmatched["am_class"].value_counts())
print()

# High-priority: no CFTR2 classification but AlphaMissense calls likely_pathogenic
flagged = unmatched[unmatched["am_class"] == "likely_pathogenic"].sort_values(
    "am_pathogenicity", ascending=False
)

print(f"Flagged as likely_pathogenic: {len(flagged)}")
print()
print(flagged[["variant", "am_variant", "am_pathogenicity"]].head(20))

flagged.to_csv("flagged_unclassified.csv", index=False)
print("\nSaved to flagged_unclassified.csv")


Unmatched variants with AM score: 2411

am_class
likely_benign        1349
likely_pathogenic     705
ambiguous             357
Name: count, dtype: int64

Flagged as likely_pathogenic: 705

         variant am_variant  am_pathogenicity
1507   Leu526Pro      L526P            0.9965
3171  Leu1242Pro     L1242P            0.9945
1646   Leu578Pro      L578P            0.9935
432     Pro99Arg       P99R            0.9933
3484  Leu1368Pro     L1368P            0.9929
3001  Arg1162Pro     R1162P            0.9923
2951  Trp1145Arg     W1145R            0.9920
2950  Trp1145Arg     W1145R            0.9920
2629  Gly1003Arg     G1003R            0.9908
2630  Gly1003Arg     G1003R            0.9908
2821  Arg1097Pro     R1097P            0.9900
1651   Asp579Val      D579V            0.9897
1006   Phe337Leu      F337L            0.9878
1008   Phe337Leu      F337L            0.9878
1007   Phe337Leu      F337L            0.9878
1027   Arg347Ser      R347S            0.9877
2688  Phe1033Leu     F1033L  

In [8]:
flagged = flagged.drop_duplicates(subset=["am_variant"]).sort_values(
    "am_pathogenicity", ascending=False
)

print(f"After dedup: {len(flagged)} unique likely_pathogenic variants")
print()
print(flagged[["variant", "am_variant", "am_pathogenicity"]].head(20))

flagged.to_csv("flagged_unclassified.csv", index=False)
print("Saved to flagged_unclassified.csv")


After dedup: 546 unique likely_pathogenic variants

         variant am_variant  am_pathogenicity
1507   Leu526Pro      L526P            0.9965
3171  Leu1242Pro     L1242P            0.9945
1646   Leu578Pro      L578P            0.9935
432     Pro99Arg       P99R            0.9933
3484  Leu1368Pro     L1368P            0.9929
3001  Arg1162Pro     R1162P            0.9923
2951  Trp1145Arg     W1145R            0.9920
2629  Gly1003Arg     G1003R            0.9908
2821  Arg1097Pro     R1097P            0.9900
1651   Asp579Val      D579V            0.9897
1006   Phe337Leu      F337L            0.9878
1027   Arg347Ser      R347S            0.9877
2688  Phe1033Leu     F1033L            0.9874
1503   Cys524Tyr      C524Y            0.9873
3425  Leu1346Pro     L1346P            0.9873
2750  Leu1062Pro     L1062P            0.9868
3168  Gly1241Asp     G1241D            0.9861
2926  Ala1136Pro     A1136P            0.9857
2911  Gly1130Arg     G1130R            0.9847
2439   Asp924His      D924H 